# ClimateVariables

Exploracion de variables climaticas disponibles via API para evaluar su utilidad en el proyecto RAIZ.

## Objetivo

Identificar, consultar y documentar variables climaticas utiles para analisis agricola: precipitacion, temperatura, humedad, presion atmosferica y otras variables agroclimaticas disponibles.

Este notebook no entrena modelos. Su proposito es revisar estructura, cobertura temporal, cobertura geografica, unidades, sensores y viabilidad de integracion.

## 1. Configuracion de entorno

En Colab se monta Google Drive para acceder a datos compartidos del equipo cuando sea necesario. Las consultas API deben funcionar sin depender de archivos locales.

Convencion de rutas:

- `DATA_ROOT`: carpeta compartida del equipo, normalmente de solo lectura.
- `PROCESSED_ROOT`: carpeta personal para guardar datasets procesados o salidas temporales.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_ROOT = Path('/content/drive/MyDrive/eco2026')
PROCESSED_ROOT = Path('/content/drive/MyDrive/eco2026_processed')

if IN_COLAB:
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print(f'IN_COLAB={IN_COLAB}')
print(f'DATA_ROOT={DATA_ROOT}')
print(f'PROCESSED_ROOT={PROCESSED_ROOT}')

## 2. Funciones comunes para APIs

Funciones auxiliares para consultar datasets de datos.gov.co usando Socrata. La idea es reutilizar el mismo patron para cada variable climatica.

In [ ]:
def consultar_datos_gov(dataset_id, select='*', where=None, limit=50000, offset=None, order=None, timeout=60):
    """Consulta un dataset publico de datos.gov.co con una consulta SoQL simple."""
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'

    query_parts = [f'SELECT {select}']
    if where:
        query_parts.append(f'WHERE {where}')
    if order:
        query_parts.append(f'ORDER BY {order}')
    query_parts.append(f'LIMIT {limit}')
    if offset is not None:
        query_parts.append(f'OFFSET {offset}')

    params = {'$query': ' '.join(query_parts)}
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    return pd.DataFrame(data)


def resumen_dataframe(df):
    """Resume rapidamente forma, columnas, tipos y nulos de un DataFrame."""
    print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}')
    display(pd.DataFrame({
        'columna': df.columns,
        'tipo': [df[col].dtype for col in df.columns],
        'nulos': [df[col].isna().sum() for col in df.columns],
        'valores_unicos': [df[col].nunique(dropna=True) for col in df.columns],
    }))


def construir_where_in(columna, valores):
    """Construye una condicion SoQL IN escapando comillas simples."""
    valores_limpios = [str(valor).replace("'", "''") for valor in valores]
    valores_sql = ', '.join([f"'{valor}'" for valor in valores_limpios])
    return f'{columna} IN ({valores_sql})'


def valores_distintos(dataset_id, columna, order=None):
    """Lista valores distintos de una columna sin descargar el dataset completo."""
    order = order or columna
    df = consultar_datos_gov(
        dataset_id,
        select=columna,
        where=f'{columna} IS NOT NULL',
        order=order,
        limit=50000
    )
    return df.drop_duplicates().sort_values(columna).reset_index(drop=True)


def total_registros(dataset_id, where=None, timeout=120):
    """Cuenta registros sin descargar el dataset completo."""
    df = consultar_datos_gov(dataset_id, select='count(*) as total_registros', where=where, limit=1, timeout=timeout)
    return int(df.loc[0, 'total_registros'])


def conteo_por_columna(dataset_id, columna, order=None, where_extra=None, timeout=120):
    """Agrupa registros por una columna usando SoQL."""
    order = order or columna
    where_parts = [f'{columna} IS NOT NULL']
    if where_extra:
        where_parts.append(f'({where_extra})')
    where = ' AND '.join(where_parts)
    df = consultar_datos_gov(
        dataset_id,
        select=f'{columna}, count(*) as total_registros',
        limit=50000,
        order=order,
        where=where,
        timeout=timeout
    )
    if 'total_registros' in df.columns:
        df['total_registros'] = pd.to_numeric(df['total_registros'], errors='coerce').astype('Int64')
    return df


def guardar_dataset_procesado(df, filename, subdir='climate_variables', formato=None):
    """Guarda un DataFrame en la carpeta personal de procesados."""
    output_dir = PROCESSED_ROOT / subdir
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / filename
    formato = formato or output_path.suffix.lower().lstrip('.')

    if formato == 'csv':
        df.to_csv(output_path, index=False)
    elif formato == 'parquet':
        df.to_parquet(output_path, index=False)
    else:
        raise ValueError("Formato soportado: 'csv' o 'parquet'")

    print(f'Dataset guardado en: {output_path}')
    return output_path

## 3. Precipitacion

Registrar aqui el dataset/API usado para precipitacion, su cobertura y sus columnas clave.

Preguntas guia:

- Que unidad usa la variable?
- La observacion es diaria, horaria, acumulada o instantanea?
- Tiene estacion, municipio, latitud y longitud?
- Desde que fecha hasta que fecha hay datos?
- Se puede agregar a municipio + año o municipio + periodo agricola?

In [ ]:
# TODO: completar con el dataset_id de precipitacion encontrado.
PRECIPITACION_DATASET_ID = ''

if PRECIPITACION_DATASET_ID:
    df_precipitacion = consultar_datos_gov(PRECIPITACION_DATASET_ID, limit=1000)
    display(df_precipitacion.head())
    resumen_dataframe(df_precipitacion)
else:
    print('Pendiente: definir PRECIPITACION_DATASET_ID')

### 3.1 Comparacion inicial de datasets de precipitacion

Datasets candidatos:

- `s54a-sgyg`
- `m84s-22dd`

Ambos parecen tener columnas equivalentes. La primera comparacion busca responder:

- Tienen las mismas columnas?
- Cuantos registros tiene cada uno?
- Que departamentos aparecen en cada uno?
- Cuantos registros hay por departamento?

Nota Socrata: para paginar descargas se puede usar `LIMIT` + `OFFSET`, pero en este notebook priorizamos consultas agregadas para evitar bajar millones de filas.

In [ ]:
PRECIPITACION_DATASETS = {
    'precipitacion_a': 's54a-sgyg',
    'precipitacion_b': 'm84s-22dd',
}

muestras_precipitacion = {}

for nombre, dataset_id in PRECIPITACION_DATASETS.items():
    print(f'== {nombre} | {dataset_id} ==')
    muestra = consultar_datos_gov(dataset_id, limit=1)
    muestras_precipitacion[nombre] = muestra
    display(muestra)
    print('Columnas:', list(muestra.columns))
    print()


In [ ]:
columnas_por_dataset = {
    nombre: set(df.columns)
    for nombre, df in muestras_precipitacion.items()
}

columnas_comunes = set.intersection(*columnas_por_dataset.values())
columnas_union = set.union(*columnas_por_dataset.values())

comparacion_columnas = pd.DataFrame({
    'columna': sorted(columnas_union),
    'en_precipitacion_a': [col in columnas_por_dataset['precipitacion_a'] for col in sorted(columnas_union)],
    'en_precipitacion_b': [col in columnas_por_dataset['precipitacion_b'] for col in sorted(columnas_union)],
})

display(comparacion_columnas)
print(f'Columnas comunes: {len(columnas_comunes)}')
print(f'Columnas totales observadas: {len(columnas_union)}')

In [ ]:
EJECUTAR_DISTINCT_DEPARTAMENTOS = False

if EJECUTAR_DISTINCT_DEPARTAMENTOS:
    departamentos_distintos = []

    for nombre, dataset_id in PRECIPITACION_DATASETS.items():
        departamentos = valores_distintos(dataset_id, 'departamento')
        departamentos['dataset'] = nombre
        departamentos['dataset_id'] = dataset_id
        departamentos_distintos.append(departamentos)

    departamentos_distintos_df = pd.concat(departamentos_distintos, ignore_index=True)
    display(departamentos_distintos_df[['dataset', 'dataset_id', 'departamento']])
else:
    departamentos_distintos_df = pd.DataFrame(columns=['dataset', 'dataset_id', 'departamento'])
    print('Distinct global desactivado: puede tardar mucho en datasets de millones de filas.')

In [ ]:
# Departamentos priorizados para el proyecto RAIZ.
DEPARTAMENTOS_INTERES = [
    'CUNDINAMARCA',
    'BOYACÁ',
    'ANTIOQUIA',
]

print(DEPARTAMENTOS_INTERES)

In [ ]:
MUESTRA_POR_DEPARTAMENTO = 1000
muestras_departamento = []

for nombre, dataset_id in PRECIPITACION_DATASETS.items():
    for departamento in DEPARTAMENTOS_INTERES:
        where_departamento = construir_where_in('departamento', [departamento])
        muestra = consultar_datos_gov(
            dataset_id,
            where=where_departamento,
            limit=MUESTRA_POR_DEPARTAMENTO,
            timeout=120
        )
        muestra['dataset'] = nombre
        muestra['dataset_id'] = dataset_id
        muestras_departamento.append(muestra)

muestras_departamento_df = pd.concat(muestras_departamento, ignore_index=True)
display(muestras_departamento_df.head())
print(f'Filas descargadas para exploracion: {len(muestras_departamento_df):,}')
display(muestras_departamento_df.groupby(['dataset', 'dataset_id', 'departamento']).size().reset_index(name='filas_muestra'))

In [ ]:
EJECUTAR_CONTEOS_FILTRADOS = False

if EJECUTAR_CONTEOS_FILTRADOS:
    conteos_departamento = []

    for nombre, dataset_id in PRECIPITACION_DATASETS.items():
        for departamento in DEPARTAMENTOS_INTERES:
            where_departamento = construir_where_in('departamento', [departamento])
            conteos_departamento.append({
                'dataset': nombre,
                'dataset_id': dataset_id,
                'departamento': departamento,
                'total_registros': total_registros(dataset_id, where=where_departamento, timeout=180),
            })

    conteos_departamento_df = pd.DataFrame(conteos_departamento)
    display(conteos_departamento_df[['dataset', 'dataset_id', 'departamento', 'total_registros']])

    comparacion_departamentos = (
        conteos_departamento_df
        .pivot_table(
            index='departamento',
            columns='dataset',
            values='total_registros',
            aggfunc='sum'
        )
        .reset_index()
    )

    comparacion_departamentos['diferencia_b_menos_a'] = (
        comparacion_departamentos.get('precipitacion_b', 0).fillna(0)
        - comparacion_departamentos.get('precipitacion_a', 0).fillna(0)
    )

    display(comparacion_departamentos.sort_values('departamento'))
else:
    print('Conteos filtrados desactivados: count(*) tambien puede tardar mucho en estos endpoints.')

### 3.2 Descarga experimental de precipitacion

Esta seccion prepara una descarga por lotes para el dataset seleccionado (`s54a-sgyg`). La prueba debe empezar con un solo departamento y un solo mes, midiendo tiempo y tamano antes de ampliar la descarga.

La salida se guarda directamente en Parquet particionado dentro de `eco2026_processed`, evitando CSV intermedios.

In [ ]:
PRECIPITACION_FUENTE_PRINCIPAL = 's54a-sgyg'

# Seguridad: mantener en False hasta decidir ejecutar la prueba en Colab.
EJECUTAR_DESCARGA_PRECIPITACION = False

# Prueba inicial recomendada: un departamento y un mes.
DESCARGA_DEPARTAMENTO = 'ANTIOQUIA'
DESCARGA_ANIO = 2026
DESCARGA_MES = 1

# Socrata pagina con LIMIT + OFFSET dentro del $query.
# Usamos 1000 como valor conservador porque muchos endpoints publicos no entregan mas por consulta.
DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = 1  # Cambiar a None para completar todo el mes.
SOBRESCRIBIR_PARQUET = False

print({
    'dataset': PRECIPITACION_FUENTE_PRINCIPAL,
    'departamento': DESCARGA_DEPARTAMENTO,
    'anio': DESCARGA_ANIO,
    'mes': DESCARGA_MES,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
})

In [ ]:
def inicio_mes_siguiente(anio, mes):
    """Devuelve el primer dia del mes siguiente."""
    if mes == 12:
        return anio + 1, 1
    return anio, mes + 1


def normalizar_precipitacion_cruda(df, dataset_id):
    """Aplica conversiones minimas antes de guardar el lote crudo."""
    df = df.copy()
    df['dataset_id'] = dataset_id

    if 'fechaobservacion' in df.columns:
        df['fechaobservacion'] = pd.to_datetime(df['fechaobservacion'], errors='coerce')
    if 'valorobservado' in df.columns:
        df['valorobservado'] = pd.to_numeric(df['valorobservado'], errors='coerce')
    for columna in ['latitud', 'longitud']:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors='coerce')

    return df


def ruta_precipitacion_parquet(dataset_id, departamento, anio, mes):
    """Construye la carpeta particionada para un mes de precipitacion."""
    departamento_particion = str(departamento).replace(' ', '_')
    return (
        PROCESSED_ROOT
        / 'precipitacion'
        / f'fuente={dataset_id}'
        / f'departamento={departamento_particion}'
        / f'anio={anio}'
        / f'mes={mes:02d}'
    )


def descargar_precipitacion_mes(
    dataset_id,
    departamento,
    anio,
    mes,
    limit=50000,
    max_lotes=1,
    sobrescribir=False,
    timeout=180,
):
    """Descarga un mes por lotes y guarda cada lote como Parquet."""
    try:
        import pyarrow  # noqa: F401
    except ImportError as exc:
        raise ImportError('Para guardar Parquet en Colab instala pyarrow: !pip install pyarrow') from exc

    import time
    from datetime import datetime

    anio_fin, mes_fin = inicio_mes_siguiente(anio, mes)
    fecha_inicio = f'{anio:04d}-{mes:02d}-01T00:00:00'
    fecha_fin = f'{anio_fin:04d}-{mes_fin:02d}-01T00:00:00'
    departamento_sql = str(departamento).replace("'", "''")

    where = (
        f"departamento = '{departamento_sql}' "
        f"AND fechaobservacion >= '{fecha_inicio}' "
        f"AND fechaobservacion < '{fecha_fin}'"
    )

    output_dir = ruta_precipitacion_parquet(dataset_id, departamento, anio, mes)
    output_dir.mkdir(parents=True, exist_ok=True)

    total_filas = 0
    lotes = []
    offset = 0
    lote_idx = 0
    inicio = time.perf_counter()
    inicio_reloj = datetime.now()
    print(f'Inicio descarga: {inicio_reloj:%Y-%m-%d %H:%M:%S}')

    while True:
        if max_lotes is not None and lote_idx >= max_lotes:
            print(f'Corte por max_lotes={max_lotes}.')
            break

        lote_inicio = time.perf_counter()
        lote_inicio_reloj = datetime.now()
        print(f'Consultando lote {lote_idx} | offset={offset:,} | limit={limit:,} | inicio={lote_inicio_reloj:%H:%M:%S}')
        df_lote = consultar_datos_gov(
            dataset_id,
            where=where,
            limit=limit,
            offset=offset,
            order='fechaobservacion, codigoestacion',
            timeout=timeout,
        )

        if df_lote.empty:
            print('La API no devolvio mas filas.')
            break

        df_lote = normalizar_precipitacion_cruda(df_lote, dataset_id)
        output_path = output_dir / f'part-{lote_idx:05d}.parquet'

        if output_path.exists() and not sobrescribir:
            print(f'Ya existe {output_path.name}; no se sobrescribe.')
        else:
            df_lote.to_parquet(output_path, index=False)

        filas_lote = len(df_lote)
        lote_fin_reloj = datetime.now()
        duracion_lote = time.perf_counter() - lote_inicio
        total_filas += filas_lote
        lotes.append({
            'lote': lote_idx,
            'offset': offset,
            'filas': filas_lote,
            'duracion_segundos': round(duracion_lote, 2),
            'inicio': lote_inicio_reloj.isoformat(timespec='seconds'),
            'fin': lote_fin_reloj.isoformat(timespec='seconds'),
            'archivo': str(output_path),
        })

        if filas_lote < limit:
            print('Ultimo lote detectado: filas_lote < limit.')
            break

        lote_idx += 1
        offset += limit

    fin_reloj = datetime.now()
    duracion = time.perf_counter() - inicio
    resumen = pd.DataFrame(lotes)
    print(f'Fin descarga: {fin_reloj:%Y-%m-%d %H:%M:%S}')
    print(f'Filas descargadas en esta corrida: {total_filas:,}')
    print(f'Duracion: {duracion:.1f} segundos')
    print(f'Carpeta salida: {output_dir}')
    return resumen


In [ ]:
if EJECUTAR_DESCARGA_PRECIPITACION:
    resumen_descarga_precipitacion = descargar_precipitacion_mes(
        dataset_id=PRECIPITACION_FUENTE_PRINCIPAL,
        departamento=DESCARGA_DEPARTAMENTO,
        anio=DESCARGA_ANIO,
        mes=DESCARGA_MES,
        limit=DESCARGA_LIMIT,
        max_lotes=DESCARGA_MAX_LOTES,
        sobrescribir=SOBRESCRIBIR_PARQUET,
    )
    display(resumen_descarga_precipitacion)
else:
    print('Descarga experimental desactivada. Cambiar EJECUTAR_DESCARGA_PRECIPITACION=True para probar en Colab.')

## 4. Temperatura

Exploracion de temperatura ambiente del aire. En exploraciones previas aparece el dataset `sbwg-7ju4`, pero conviene validar cobertura, columnas y formato de fecha antes de usarlo como fuente definitiva.

In [ ]:
TEMPERATURA_DATASET_ID = 'sbwg-7ju4'

df_temperatura = consultar_datos_gov(TEMPERATURA_DATASET_ID, limit=1000)
display(df_temperatura.head())
resumen_dataframe(df_temperatura)

## 5. Humedad

Registrar aqui datasets candidatos de humedad relativa, humedad del suelo u otras variables asociadas.

In [ ]:
# TODO: completar con dataset_id de humedad.
HUMEDAD_DATASET_ID = ''

if HUMEDAD_DATASET_ID:
    df_humedad = consultar_datos_gov(HUMEDAD_DATASET_ID, limit=1000)
    display(df_humedad.head())
    resumen_dataframe(df_humedad)
else:
    print('Pendiente: definir HUMEDAD_DATASET_ID')

## 6. Presion atmosferica

Registrar aqui el dataset de presion atmosferica y validar si aporta informacion util para rendimiento agricola o si se conserva solo como variable complementaria.

In [ ]:
# TODO: completar con dataset_id de presion atmosferica.
PRESION_DATASET_ID = ''

if PRESION_DATASET_ID:
    df_presion = consultar_datos_gov(PRESION_DATASET_ID, limit=1000)
    display(df_presion.head())
    resumen_dataframe(df_presion)
else:
    print('Pendiente: definir PRESION_DATASET_ID')

## 7. Otras variables agroclimaticas

Espacio para explorar variables adicionales: radiacion solar, brillo solar, velocidad del viento, eventos extremos, indices Niño/Niña u otras fuentes relevantes.

In [ ]:
variables_adicionales = []

# Ejemplo de registro manual:
# variables_adicionales.append({
#     'variable': 'radiacion_solar',
#     'dataset_id': 'pendiente',
#     'fuente': 'datos.gov.co',
#     'estado': 'por revisar',
# })

pd.DataFrame(variables_adicionales)

## 8. Tabla resumen de variables

Tabla de decision para comparar rapidamente las variables encontradas.

In [ ]:
resumen_variables = pd.DataFrame(columns=[
    'variable',
    'dataset_id',
    'fuente',
    'unidad',
    'frecuencia_observada',
    'nivel_geografico',
    'fecha_min',
    'fecha_max',
    'columnas_clave',
    'riesgos',
    'decision'
])

resumen_variables

## 9. Conclusiones preliminares

Anotar aqui hallazgos, dudas y decisiones. La salida esperada no es un modelo, sino una recomendacion sobre que variables climaticas pasan a la siguiente fase de integracion.